# IBGE ETL — SIDRA API + PySpark

Pipeline for extracting, normalizing, and publishing Brazilian public
indicators from IBGE.

The shared SIDRA API structure is normalized once, while each indicator keeps
only its specific business rules.


## 1. Imports, Paths, and Parameters

`ATUALIZAR_DADOS = False` reuses the most recent JSON files from the `raw`
layer. Set it to `True` only when you want to request fresh data from the API.


In [ ]:
from datetime import datetime
from pathlib import Path
from urllib.parse import urlencode

import json
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import MetaData, Table, URL, create_engine
from sqlalchemy.dialects.postgresql import insert as pg_insert
from pyspark.sql import SparkSession, functions as F, types as T
from sqlalchemy.engine import URL


def localizar_diretorio_ibge():
    cwd = Path.cwd().resolve()
    candidatos = [
        cwd,
        cwd.parent,
        cwd / "APIs" / "IBGE",
        cwd.parent / "APIs" / "IBGE",
    ]
    for candidato in candidatos:
        if (candidato / "src").is_dir() and (candidato / "data").is_dir():
            return candidato
    raise FileNotFoundError(
        "Não foi possível localizar APIs/IBGE. "
        "Execute o notebook a partir do projeto ou de APIs/IBGE/src."
    )


BASE_URL = "https://servicodados.ibge.gov.br/api/v3/agregados"
IBGE_DIR = localizar_diretorio_ibge()
RAW_DIR = IBGE_DIR / "data" / "raw"
PROJECT_DIR = IBGE_DIR.parents[1]
PROCESSED_DIR = PROJECT_DIR / "dados" / "IBGE"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ATUALIZAR_DADOS = False

# True publishes each final dataset to PostgreSQL using an UPSERT.
PUBLICAR_POSTGRES = True
POSTGRES_SCHEMA = "analytics"

IDADES_QUINQUENAIS = (
    "93070,93084,93085,93086,93087,93088,93089,93090,93091,"
    "93092,93093,93094,93095,93096,93097,93098,49108,49109,"
    "60040,60041,6653"
)
GRUPOS_IPCA = "7169,7170,7445,7486,7558,7625,7660,7712,7766,7786"

INDICADORES = {
    "populacao": {
        "agregado": "6579",
        "periodos": "2012-2021",
        "variaveis": "9324",
        "localidades": "N6[all]",
        "classificacao": None,
    },
    "pib": {
        "agregado": "6784",
        "periodos": "2012-2021",
        "variaveis": "all",
        "localidades": "N1[all]",
        "classificacao": None,
    },
    "faixa_etaria": {
        "agregado": "9514",
        "periodos": "2022",
        "variaveis": "93",
        "localidades": "N3[all]",
        "classificacao": (
            f"2[6794,4,5]|287[{IDADES_QUINQUENAIS}]|286[113635]"
        ),
    },
    "desemprego": {
        "agregado": "4099",
        "periodos": "all",
        "variaveis": "4099,4118",
        "localidades": "N3[all]",
        "classificacao": None,
    },
    "ipca": {
        "agregado": "7060",
        "periodos": "all",
        "variaveis": "63,69,2265,66",
        "localidades": "N7[all]",
        "classificacao": f"315[{GRUPOS_IPCA}]",
    },
}

INDICADORES_ATIVOS = list(INDICADORES)

print("Diretório IBGE:", IBGE_DIR)
print("Indicadores ativos:", INDICADORES_ATIVOS)


Diretório IBGE: C:\Users\Gabriel\Desktop\Python_Treinamento\projeto-indicadores-publicos\APIs\IBGE
Indicadores ativos: ['populacao', 'pib', 'faixa_etaria', 'desemprego', 'ipca']


In [ ]:
load_dotenv(IBGE_DIR / ".env")

postgres_url = URL.create(
    drivername="postgresql+psycopg",
    username=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    host=os.getenv("POSTGRES_HOST"),
    port=int(os.getenv("POSTGRES_PORT", "5432")),
    database=os.getenv("POSTGRES_DB"),
)

postgres_engine = create_engine(
    postgres_url,
    pool_pre_ping=True,
)

## 2. Spark Session and SIDRA API Schema


In [100]:
spark = (
    SparkSession.builder
    .appName("ETL-IBGE-SIDRA")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

categoria_schema = T.MapType(T.StringType(), T.StringType())

classificacao_schema = T.StructType([
    T.StructField("id", T.StringType()),
    T.StructField("nome", T.StringType()),
    T.StructField("categoria", categoria_schema),
])

localidade_schema = T.StructType([
    T.StructField("id", T.StringType()),
    T.StructField(
        "nivel",
        T.StructType([
            T.StructField("id", T.StringType()),
            T.StructField("nome", T.StringType()),
        ]),
    ),
    T.StructField("nome", T.StringType()),
])

serie_schema = T.StructType([
    T.StructField("localidade", localidade_schema),
    T.StructField(
        "serie",
        T.MapType(T.StringType(), T.StringType()),
    ),
])

resultado_schema = T.StructType([
    T.StructField(
        "classificacoes",
        T.ArrayType(classificacao_schema),
    ),
    T.StructField("series", T.ArrayType(serie_schema)),
])

variavel_schema = T.StructType([
    T.StructField("id", T.StringType()),
    T.StructField("variavel", T.StringType()),
    T.StructField("unidade", T.StringType()),
    T.StructField(
        "resultados",
        T.ArrayType(resultado_schema),
    ),
])

sidra_schema = T.ArrayType(variavel_schema)


## 3. Data Extraction and Metadata Discovery


In [101]:
def consultar_api(url, tentativas=3, timeout=60):
    for tentativa in range(1, tentativas + 1):
        try:
            resposta = requests.get(url, timeout=timeout)
            resposta.raise_for_status()
            return resposta.json()
        except requests.RequestException:
            if tentativa == tentativas:
                raise
            time.sleep(3 * tentativa)


def descobrir_agregado(codigo):
    meta = consultar_api(f"{BASE_URL}/{codigo}/metadados")
    print(f"Tabela {codigo}: {meta['nome']}")
    print("\nNíveis territoriais:", meta.get("nivelTerritorial", {}))
    print("\nVariáveis:")
    for item in meta.get("variaveis", []):
        print(
            f"  {item['id']}: {item['nome']} "
            f"[{item.get('unidade', '')}]"
        )
    print("\nClassificações e categorias:")
    for classe in meta.get("classificacoes", []):
        print(f"  {classe['id']}: {classe['nome']}")
        for item in classe.get("categorias", []):
            print(f"    {item['id']}: {item['nome']}")
    return meta


def montar_url(config):
    query = {"localidades": config["localidades"]}
    if config.get("classificacao"):
        query["classificacao"] = config["classificacao"]
    return (
        f"{BASE_URL}/{config['agregado']}"
        f"/periodos/{config['periodos']}"
        f"/variaveis/{config['variaveis']}"
        f"?{urlencode(query, safe='[],|')}"
    )


def extrair_indicador(nome, config):
    url = montar_url(config)
    print(f"Extraindo {nome}: {url}")
    dados = consultar_api(url)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    destino = (
        RAW_DIR
        / f"ibge_{nome}_{config['agregado']}_{timestamp}.json"
    )
    with destino.open("w", encoding="utf-8") as arquivo:
        json.dump(dados, arquivo, ensure_ascii=False, indent=2)
    print("Salvo:", destino)
    return destino


def json_mais_recente(nome, agregado):
    padroes = [f"ibge_{nome}_{agregado}_*.json"]
    if nome in {"populacao", "pib"}:
        padroes.append(f"ibge_agregado_{agregado}_*.json")

    arquivos = [
        arquivo
        for padrao in padroes
        for arquivo in RAW_DIR.glob(padrao)
    ]
    if not arquivos:
        raise FileNotFoundError(
            f"Não existe JSON bruto para {nome}. "
            "Use ATUALIZAR_DADOS = True para consultar a API."
        )
    return max(arquivos, key=lambda caminho: caminho.stat().st_mtime)


arquivos_raw = {}
for nome in INDICADORES_ATIVOS:
    config = INDICADORES[nome]
    arquivos_raw[nome] = (
        extrair_indicador(nome, config)
        if ATUALIZAR_DADOS
        else json_mais_recente(nome, config["agregado"])
    )

arquivos_raw


{'populacao': WindowsPath('C:/Users/Gabriel/Desktop/Python_Treinamento/projeto-indicadores-publicos/APIs/IBGE/data/raw/ibge_agregado_6579_20260728_104316.json'),
 'pib': WindowsPath('C:/Users/Gabriel/Desktop/Python_Treinamento/projeto-indicadores-publicos/APIs/IBGE/data/raw/ibge_agregado_6784_20260728_104317.json'),
 'faixa_etaria': WindowsPath('C:/Users/Gabriel/Desktop/Python_Treinamento/projeto-indicadores-publicos/APIs/IBGE/data/raw/ibge_faixa_etaria_9514_20260728_105028.json'),
 'desemprego': WindowsPath('C:/Users/Gabriel/Desktop/Python_Treinamento/projeto-indicadores-publicos/APIs/IBGE/data/raw/ibge_desemprego_4099_20260728_105032.json'),
 'ipca': WindowsPath('C:/Users/Gabriel/Desktop/Python_Treinamento/projeto-indicadores-publicos/APIs/IBGE/data/raw/ibge_ipca_7060_20260728_105037.json')}

## 4. Shared Transformation Functions

`normalizar_sidra` produces the common long-format layer used by every
indicator. Special API values, such as `...`, are converted to null.


In [102]:
PADRAO_NUMERICO = r"^[-+]?\d+(?:[.,]\d+)?$"


def normalizar_sidra(caminho):
    return (
        spark.read
        .text(str(caminho), wholetext=True)
        .select(
            F.explode(
                F.from_json("value", sidra_schema)
            ).alias("variavel")
        )
        .select(
            F.col("variavel.id")
                .cast("int")
                .alias("variavel_id"),
            F.col("variavel.variavel")
                .alias("variavel_nome"),
            F.col("variavel.unidade")
                .alias("unidade"),
            F.explode("variavel.resultados")
                .alias("resultado"),
        )
        .select(
            "variavel_id",
            "variavel_nome",
            "unidade",
            F.col("resultado.classificacoes")
                .alias("classificacoes"),
            F.explode("resultado.series")
                .alias("serie_item"),
        )
        .select(
            "variavel_id",
            "variavel_nome",
            "unidade",
            "classificacoes",
            F.col("serie_item.localidade.id")
                .alias("localidade_id"),
            F.col("serie_item.localidade.nome")
                .alias("localidade_nome"),
            F.col("serie_item.localidade.nivel.id")
                .alias("nivel_id"),
            F.col("serie_item.localidade.nivel.nome")
                .alias("nivel_nome"),
            F.explode("serie_item.serie")
                .alias("periodo", "valor_original"),
        )
        .withColumn(
            "valor",
            F.when(
                F.trim("valor_original").rlike(PADRAO_NUMERICO),
                F.regexp_replace(
                    F.trim("valor_original"),
                    ",",
                    ".",
                ).cast("double"),
            ),
        )
    )


def extrair_categoria(classificacao_id):
    classificacao = F.element_at(
        F.filter(
            F.col("classificacoes"),
            lambda item: (
                item["id"] == F.lit(str(classificacao_id))
            ),
        ),
        1,
    )
    return F.element_at(
        F.map_entries(classificacao["categoria"]),
        1,
    )


def adicionar_data_anual(df):
    return (
        df
        .withColumn("ano", F.col("periodo").cast("int"))
        .withColumn(
            "data_referencia",
            F.make_date("ano", F.lit(1), F.lit(1)),
        )
    )


def adicionar_data_mensal(df):
    return (
        df
        .withColumn(
            "ano",
            F.substring("periodo", 1, 4).cast("int"),
        )
        .withColumn(
            "mes",
            F.substring("periodo", 5, 2).cast("int"),
        )
        .withColumn(
            "data_referencia",
            F.make_date("ano", "mes", F.lit(1)),
        )
    )


def adicionar_data_trimestral(df):
    return (
        df
        .withColumn(
            "ano",
            F.substring("periodo", 1, 4).cast("int"),
        )
        .withColumn(
            "trimestre_num",
            F.substring("periodo", 5, 2).cast("int"),
        )
        .withColumn(
            "trimestre",
            F.concat(
                "trimestre_num",
                F.lit("º trimestre"),
            ),
        )
        .withColumn(
            "data_referencia",
            F.make_date(
                "ano",
                (F.col("trimestre_num") - 1) * 3 + 1,
                F.lit(1),
            ),
        )
    )


def pivotar_variaveis(df, chaves, variaveis, valor="valor"):
    resultado = (
        df
        .groupBy(*chaves)
        .pivot("variavel_id", list(variaveis))
        .agg(F.first(valor))
    )
    for variavel_id, nome_coluna in variaveis.items():
        resultado = resultado.withColumnRenamed(
            str(variavel_id),
            nome_coluna,
        )
    return resultado


_postgres_engine = None
_postgres_preparado = False


def obter_engine_postgres():
    global _postgres_engine

    if _postgres_engine is not None:
        return _postgres_engine

    project_dir = IBGE_DIR.parents[1]
    load_dotenv(project_dir / ".env")

    required_settings = [
        "POSTGRES_HOST",
        "POSTGRES_DB",
        "POSTGRES_USER",
        "POSTGRES_PASSWORD",
    ]
    missing_settings = [
        setting
        for setting in required_settings
        if not os.getenv(setting)
    ]
    if missing_settings:
        raise RuntimeError(
            "Missing PostgreSQL settings in .env: "
            + ", ".join(missing_settings)
        )

    database_url = URL.create(
        drivername="postgresql+psycopg",
        username=os.environ["POSTGRES_USER"],
        password=os.environ["POSTGRES_PASSWORD"],
        host=os.environ["POSTGRES_HOST"],
        port=int(os.getenv("POSTGRES_PORT", "5432")),
        database=os.environ["POSTGRES_DB"],
    )
    _postgres_engine = create_engine(
        database_url,
        pool_pre_ping=True,
    )
    return _postgres_engine


def preparar_postgres():
    global _postgres_preparado

    if _postgres_preparado:
        return

    project_dir = IBGE_DIR.parents[1]
    ddl_path = (
        project_dir
        / "database"
        / "01_create_analytics_tables.sql"
    )
    ddl = ddl_path.read_text(encoding="utf-8")
    statements = [
        statement.strip()
        for statement in ddl.split(";")
        if statement.strip()
    ]

    engine = obter_engine_postgres()
    with engine.begin() as connection:
        for statement in statements:
            connection.exec_driver_sql(statement)

    _postgres_preparado = True


def publicar_postgres(pandas_df, tabela, chave, chunk_size=1000):
    if not chave:
        raise ValueError(
            f"{tabela}: an UPSERT requires a primary key"
        )

    preparar_postgres()
    engine = obter_engine_postgres()

    dataframe = (
        pandas_df
        .astype(object)
        .where(pd.notna(pandas_df), None)
    )
    records = dataframe.to_dict(orient="records")

    with engine.begin() as connection:
        metadata = MetaData()
        target = Table(
            tabela,
            metadata,
            schema=POSTGRES_SCHEMA,
            autoload_with=connection,
        )

        table_columns = {column.name for column in target.columns}
        csv_columns = set(dataframe.columns)
        if csv_columns != table_columns:
            raise ValueError(
                f"{tabela}: CSV columns do not match PostgreSQL. "
                f"Missing: {sorted(table_columns - csv_columns)}; "
                f"unexpected: {sorted(csv_columns - table_columns)}"
            )

        for offset in range(0, len(records), chunk_size):
            chunk = records[offset:offset + chunk_size]
            statement = pg_insert(target).values(chunk)
            update_values = {
                column.name: statement.excluded[column.name]
                for column in target.columns
                if column.name not in chave
            }
            statement = statement.on_conflict_do_update(
                index_elements=chave,
                set_=update_values,
            )
            connection.execute(statement)

    print(
        f"{POSTGRES_SCHEMA}.{tabela}: "
        f"{len(dataframe):,} rows inserted or updated"
    )


def publicar_dados(df, nome, chave):
    destino = PROCESSED_DIR / f"{nome}.csv"
    destino.parent.mkdir(parents=True, exist_ok=True)

    pandas_df = df.toPandas()
    duplicadas = int(
        pandas_df.duplicated(subset=chave).sum()
    )
    if duplicadas:
        raise ValueError(
            f"{nome}: {duplicadas} duplicate keys in {chave}"
        )

    pandas_df.to_csv(
        destino,
        index=False,
        encoding="utf-8-sig",
    )
    print(f"{nome}: {len(pandas_df):,} rows -> {destino}")

    if PUBLICAR_POSTGRES:
        publicar_postgres(
            pandas_df=pandas_df,
            tabela=nome.lower(),
            chave=chave,
        )

    return destino


## 5. State Dimension


In [103]:
ufs = [
    (11, "RO", "Rondônia", "Norte"),
    (12, "AC", "Acre", "Norte"),
    (13, "AM", "Amazonas", "Norte"),
    (14, "RR", "Roraima", "Norte"),
    (15, "PA", "Pará", "Norte"),
    (16, "AP", "Amapá", "Norte"),
    (17, "TO", "Tocantins", "Norte"),
    (21, "MA", "Maranhão", "Nordeste"),
    (22, "PI", "Piauí", "Nordeste"),
    (23, "CE", "Ceará", "Nordeste"),
    (24, "RN", "Rio Grande do Norte", "Nordeste"),
    (25, "PB", "Paraíba", "Nordeste"),
    (26, "PE", "Pernambuco", "Nordeste"),
    (27, "AL", "Alagoas", "Nordeste"),
    (28, "SE", "Sergipe", "Nordeste"),
    (29, "BA", "Bahia", "Nordeste"),
    (31, "MG", "Minas Gerais", "Sudeste"),
    (32, "ES", "Espírito Santo", "Sudeste"),
    (33, "RJ", "Rio de Janeiro", "Sudeste"),
    (35, "SP", "São Paulo", "Sudeste"),
    (41, "PR", "Paraná", "Sul"),
    (42, "SC", "Santa Catarina", "Sul"),
    (43, "RS", "Rio Grande do Sul", "Sul"),
    (50, "MS", "Mato Grosso do Sul", "Centro-Oeste"),
    (51, "MT", "Mato Grosso", "Centro-Oeste"),
    (52, "GO", "Goiás", "Centro-Oeste"),
    (53, "DF", "Distrito Federal", "Centro-Oeste"),
]

dim_uf = spark.createDataFrame(
    ufs,
    ["uf_id", "uf_sigla", "uf_nome", "regiao"],
)

publicar_dados(dim_uf, "dim_uf", chave=["uf_id"])


dim_uf: 27 linhas -> C:\Users\Gabriel\Desktop\Python_Treinamento\projeto-indicadores-publicos\APIs\IBGE\data\processed\dim_uf.csv


WindowsPath('C:/Users/Gabriel/Desktop/Python_Treinamento/projeto-indicadores-publicos/APIs/IBGE/data/processed/dim_uf.csv')

## 6. Municipal Population


In [104]:
base_populacao = adicionar_data_anual(
    normalizar_sidra(arquivos_raw["populacao"])
)

fato_populacao_municipio = (
    base_populacao
    .withColumn(
        "municipio_id",
        F.col("localidade_id").cast("int"),
    )
    .withColumn(
        "uf_id",
        F.substring("localidade_id", 1, 2).cast("int"),
    )
    .withColumn(
        "municipio_nome",
        F.regexp_replace(
            "localidade_nome",
            r"\s*-\s*[A-Z]{2}$",
            "",
        ),
    )
    .withColumn("populacao", F.col("valor").cast("long"))
    .select(
        "data_referencia",
        "ano",
        "municipio_id",
        "municipio_nome",
        "uf_id",
        "populacao",
    )
)

publicar_dados(
    fato_populacao_municipio,
    "populacao_municipio",
    chave=["data_referencia", "municipio_id"],
)


populacao_municipio: 55,710 linhas -> C:\Users\Gabriel\Desktop\Python_Treinamento\projeto-indicadores-publicos\APIs\IBGE\data\processed\populacao_municipio.csv


WindowsPath('C:/Users/Gabriel/Desktop/Python_Treinamento/projeto-indicadores-publicos/APIs/IBGE/data/processed/populacao_municipio.csv')

## 7. GDP and National Accounts


In [105]:
base_pib = adicionar_data_anual(
    normalizar_sidra(arquivos_raw["pib"])
)

colunas_pib = [
    "variavel_id",
    "variavel_nome",
    "unidade",
    "data_referencia",
    "ano",
]

fato_pib_pessoas = (
    base_pib
    .filter(F.col("variavel_id") == 93)
    .withColumn(
        "populacao_mil_pessoas",
        F.col("valor").cast("long"),
    )
    .select(*colunas_pib, "populacao_mil_pessoas")
)

fato_pib_financeiro = (
    base_pib
    .filter(
        F.col("variavel_id").isin(
            [9808, 9809, 9812, 9813]
        )
    )
    .withColumn(
        "valor_financeiro",
        F.col("valor").cast("decimal(20,2)"),
    )
    .select(*colunas_pib, "valor_financeiro")
)

fato_pib_percentual = (
    base_pib
    .filter(
        F.col("variavel_id").isin(
            [9810, 9811, 9814]
        )
    )
    .withColumn(
        "valor_percentual",
        F.col("valor").cast("decimal(10,2)"),
    )
    .select(*colunas_pib, "valor_percentual")
)

publicar_dados(
    fato_pib_pessoas,
    "PIB_pessoas",
    chave=["data_referencia", "variavel_id"],
)
publicar_dados(
    fato_pib_financeiro,
    "PIB_financeiro",
    chave=["data_referencia", "variavel_id"],
)
publicar_dados(
    fato_pib_percentual,
    "PIB_percentual",
    chave=["data_referencia", "variavel_id"],
)


PIB_pessoas: 10 linhas -> C:\Users\Gabriel\Desktop\Python_Treinamento\projeto-indicadores-publicos\APIs\IBGE\data\processed\PIB_pessoas.csv
PIB_financeiro: 40 linhas -> C:\Users\Gabriel\Desktop\Python_Treinamento\projeto-indicadores-publicos\APIs\IBGE\data\processed\PIB_financeiro.csv
PIB_percentual: 30 linhas -> C:\Users\Gabriel\Desktop\Python_Treinamento\projeto-indicadores-publicos\APIs\IBGE\data\processed\PIB_percentual.csv


WindowsPath('C:/Users/Gabriel/Desktop/Python_Treinamento/projeto-indicadores-publicos/APIs/IBGE/data/processed/PIB_percentual.csv')

## 8. Population by Age Group and State

The source contains `Total`, `Men`, and `Women`. In Power BI, do not aggregate
`Total` together with both sexes, as this would double-count the population.


In [106]:
base_faixa_etaria = (
    adicionar_data_anual(
        normalizar_sidra(arquivos_raw["faixa_etaria"])
    )
    .withColumn("categoria_sexo", extrair_categoria(2))
    .withColumn("categoria_idade", extrair_categoria(287))
)

inicio_faixa = F.regexp_extract(
    F.col("categoria_idade.value"),
    r"^(\d+)",
    1,
).cast("int")

fato_faixa_etaria = (
    base_faixa_etaria
    .withColumn(
        "uf_id",
        F.col("localidade_id").cast("int"),
    )
    .withColumn(
        "sexo_id",
        F.col("categoria_sexo.key").cast("int"),
    )
    .withColumn(
        "sexo",
        F.col("categoria_sexo.value"),
    )
    .withColumn(
        "faixa_etaria_id",
        F.col("categoria_idade.key").cast("int"),
    )
    .withColumn(
        "faixa_etaria",
        F.col("categoria_idade.value"),
    )
    .withColumn("faixa_etaria_ordem", inicio_faixa)
    .withColumn(
        "faixa_macro",
        F.when(inicio_faixa <= 14, "0 a 14 anos")
        .when(inicio_faixa <= 29, "15 a 29 anos")
        .when(inicio_faixa <= 59, "30 a 59 anos")
        .otherwise("60 anos ou mais"),
    )
    .withColumn("populacao", F.col("valor").cast("long"))
    .select(
        "data_referencia",
        "ano",
        "uf_id",
        "sexo_id",
        "sexo",
        "faixa_etaria_id",
        "faixa_etaria",
        "faixa_etaria_ordem",
        "faixa_macro",
        "populacao",
    )
)

publicar_dados(
    fato_faixa_etaria,
    "faixa_etaria_uf",
    chave=[
        "data_referencia",
        "uf_id",
        "sexo_id",
        "faixa_etaria_id",
    ],
)


faixa_etaria_uf: 1,701 linhas -> C:\Users\Gabriel\Desktop\Python_Treinamento\projeto-indicadores-publicos\APIs\IBGE\data\processed\faixa_etaria_uf.csv


WindowsPath('C:/Users/Gabriel/Desktop/Python_Treinamento/projeto-indicadores-publicos/APIs/IBGE/data/processed/faixa_etaria_uf.csv')

## 9. Quarterly Unemployment and Labor Underutilization by State


In [107]:
base_desemprego = (
    adicionar_data_trimestral(
        normalizar_sidra(arquivos_raw["desemprego"])
    )
    .withColumn(
        "uf_id",
        F.col("localidade_id").cast("int"),
    )
)

fato_desemprego = pivotar_variaveis(
    base_desemprego,
    chaves=[
        "data_referencia",
        "ano",
        "trimestre_num",
        "trimestre",
        "uf_id",
    ],
    variaveis={
        4099: "taxa_desocupacao_pct",
        4118: "taxa_subutilizacao_pct",
    },
)

publicar_dados(
    fato_desemprego,
    "desemprego_uf_trimestre",
    chave=["data_referencia", "uf_id"],
)


desemprego_uf_trimestre: 1,539 linhas -> C:\Users\Gabriel\Desktop\Python_Treinamento\projeto-indicadores-publicos\APIs\IBGE\data\processed\desemprego_uf_trimestre.csv


WindowsPath('C:/Users/Gabriel/Desktop/Python_Treinamento/projeto-indicadores-publicos/APIs/IBGE/data/processed/desemprego_uf_trimestre.csv')

## 10. Monthly IPCA by Surveyed Area

`N7` locations represent surveyed areas or metropolitan regions and must not
be interpreted as totals for their respective states.


In [108]:
base_ipca = (
    adicionar_data_mensal(
        normalizar_sidra(arquivos_raw["ipca"])
    )
    .withColumn(
        "categoria_grupo",
        extrair_categoria(315),
    )
    .withColumn(
        "grupo_ipca_id",
        F.col("categoria_grupo.key").cast("int"),
    )
    .withColumn(
        "grupo_ipca_original",
        F.col("categoria_grupo.value"),
    )
    .withColumn(
        "area_ipca_id",
        F.col("localidade_id").cast("int"),
    )
    .withColumn(
        "area_ipca",
        F.col("localidade_nome"),
    )
    .withColumn(
        "grupo_ipca_ordem",
        F.when(
            F.col("grupo_ipca_id") == 7169,
            F.lit(0),
        ).otherwise(
            F.regexp_extract(
                "grupo_ipca_original",
                r"^(\d+)",
                1,
            ).cast("int")
        ),
    )
    .withColumn(
        "grupo_ipca",
        F.when(
            F.col("grupo_ipca_id") == 7169,
            F.lit("Índice geral"),
        ).otherwise(
            F.regexp_replace(
                "grupo_ipca_original",
                r"^\d+\.\s*",
                "",
            )
        ),
    )
)

fato_ipca = pivotar_variaveis(
    base_ipca,
    chaves=[
        "data_referencia",
        "ano",
        "mes",
        "area_ipca_id",
        "area_ipca",
        "nivel_id",
        "grupo_ipca_id",
        "grupo_ipca",
        "grupo_ipca_ordem",
    ],
    variaveis={
        63: "variacao_mensal_pct",
        69: "acumulado_ano_pct",
        2265: "acumulado_12_meses_pct",
        66: "peso_mensal_pct",
    },
)

publicar_dados(
    fato_ipca,
    "ipca_area_mes",
    chave=[
        "data_referencia",
        "area_ipca_id",
        "grupo_ipca_id",
    ],
)


ipca_area_mes: 7,800 linhas -> C:\Users\Gabriel\Desktop\Python_Treinamento\projeto-indicadores-publicos\APIs\IBGE\data\processed\ipca_area_mes.csv


WindowsPath('C:/Users/Gabriel/Desktop/Python_Treinamento/projeto-indicadores-publicos/APIs/IBGE/data/processed/ipca_area_mes.csv')

## 11. Final Validations


In [109]:
validacoes = {
    "dim_uf": dim_uf.count(),
    "populacao_municipio": fato_populacao_municipio.count(),
    "PIB_pessoas": fato_pib_pessoas.count(),
    "PIB_financeiro": fato_pib_financeiro.count(),
    "PIB_percentual": fato_pib_percentual.count(),
    "faixa_etaria_uf": fato_faixa_etaria.count(),
    "desemprego_uf_trimestre": fato_desemprego.count(),
    "ipca_area_mes": fato_ipca.count(),
}

for nome, quantidade in validacoes.items():
    print(f"{nome}: {quantidade:,} linhas")

fato_ipca.select(
    *[
        F.sum(
            F.col(coluna).isNull().cast("int")
        ).alias(coluna)
        for coluna in [
            "variacao_mensal_pct",
            "acumulado_ano_pct",
            "acumulado_12_meses_pct",
            "peso_mensal_pct",
        ]
    ]
).show()


dim_uf: 27 linhas
populacao_municipio: 55,710 linhas
PIB_pessoas: 10 linhas
PIB_financeiro: 40 linhas
PIB_percentual: 30 linhas
faixa_etaria_uf: 1,701 linhas
desemprego_uf_trimestre: 1,539 linhas
ipca_area_mes: 7,800 linhas
+-------------------+-----------------+----------------------+---------------+
|variacao_mensal_pct|acumulado_ano_pct|acumulado_12_meses_pct|peso_mensal_pct|
+-------------------+-----------------+----------------------+---------------+
|                  0|                0|                  1100|              0|
+-------------------+-----------------+----------------------+---------------+

